In [1]:
import os
import sys
import time

import joblib
import skfda
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from skfda.preprocessing.dim_reduction import FPCA
from sklearn.svm import OneClassSVM

sys.path.insert(0, '../../..')
from utils.helper_functions import normalize_series
from utils.metrics import show_feature_extraction_metrics
from utils.plotting import plot_movement_data
from utils.constants import Configs

TARGET_MOVEMENT = 'Pick'

/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


In [2]:
# Load the processed data from the csv
cache_path = Configs.ICATER_ALL_SENSORS_FILTERED_DATA_DIR
if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Processed data not found at {cache_path}. Please run the data processing notebook first.")

df = pd.read_csv(cache_path)

Loading processed data from ../../data/processed/filtered_data.csv


In [ ]:
# Load the known outliers from the csv
outliers_csv = pd.read_csv(Configs.ICATER_OUTLIER_FILES).drop_duplicates(subset=['sub', 'repID'])
total_known_outliers = len(outliers_csv)

print(f"Total known outliers: {total_known_outliers}")
outliers_csv.head()

Total known outliers: 14


,sub,repID
0,2,2
1,4,1
2,5,2
3,5,5
4,7,1


In [6]:
group_array = ['xs', 'ys', 'zs', 'xe', 'ye', 'ze', 'xw', 'yw', 'zw']
sample_size = None

# Usar o TimeSeriesScalerMeanVariance para normalizar as séries temporais,
# mas sem escalar para zero média e unidade de variância (useScaler=False)
X_movement = normalize_series(
        df_input=df,
        movLabel_filter=TARGET_MOVEMENT,
        useScaler=False,
        group_array=group_array,
        sample_size=sample_size
)  # shape (n_samples, T, n_features)

Processing time series: 100%|██████████| 380/380 [00:00<00:00, 4298.10it/s]

Shape of first series: (111, 9)


Shape of formatted series: (380, 241, 9)


## fPCA

In [7]:
print(f"\n--- A preparar dados para fPCA ({TARGET_MOVEMENT}) ---")

# 1. Extrair as coordenadas da mão (n_samples, n_timepoints, 3)
X_hand = X_movement[:, :, 6:9].copy() # .copy() para não alterar o array original

# 2. LIMPEZA DE NaNs (Forward Fill)
# Vamos percorrer cada amostra e cada eixo (X, Y, Z) e estender o último valor válido
for i in range(X_hand.shape[0]):
    for j in range(X_hand.shape[2]):
        # Usamos o Pandas temporariamente porque ele tem a melhor função de preenchimento
        s = pd.Series(X_hand[i, :, j])
        # ffill() preenche para a frente, bfill() garante que se houver NaNs no início também são tratados
        X_hand[i, :, j] = s.ffill().bfill().values

# 3. O Truque da Concatenação (Esmagar para 2D)
n_samples, n_timepoints, n_dims = X_hand.shape
X_hand_concat = np.transpose(X_hand, (0, 2, 1)).reshape(n_samples, n_timepoints * n_dims)

time_points_concat = np.arange(X_hand_concat.shape[1])

# 4. Converter para o formato FDataGrid do scikit-fda (Agora sem NaNs!)
fd = skfda.FDataGrid(data_matrix=X_hand_concat, grid_points=time_points_concat)


--- A preparar dados para fPCA (Pick) ---


In [8]:
# --- APLICAR O fPCA ---

start_time_fpca = time.perf_counter()

n_components = 5 
fpca = FPCA(n_components=n_components)

fpc_scores = fpca.fit_transform(fd)

end_time_fpca = time.perf_counter()
print(f"fPCA concluído em {end_time_fpca - start_time_fpca:.4f} segundos!")
print(f"Variância explicada pelos {n_components} componentes: {sum(fpca.explained_variance_ratio_):.2%}")

fPCA concluído em 0.2257 segundos!
Variância explicada pelos 5 componentes: 91.03%


In [ ]:
print("\n--- Iniciando Grid Search Dinâmico (fPCA + One-Class SVM) ---")

# (Aqui assume-se que o results_temp_base já foi recriado como vimos antes)
total_trials = len(results_temp_base)
known_outlier_rate = total_known_outliers / total_trials
percentagem_conhecida = known_outlier_rate * 100

min_nu = 0.01 
max_nu = known_outlier_rate + 0.10            

nu_range = [round(n, 3) for n in np.linspace(min_nu, max_nu, 15)]

# Testar valores de Gamma (kernel coefficient)
gamma_options = [0.01, 0.1, 0.5, 1.0, 'scale', 'auto']

results_log = []

for nu in nu_range:
    for g in gamma_options:
        
        # CORREÇÃO: Passar o gamma e o nu para dentro do modelo!
        model = OneClassSVM(nu=nu, kernel='rbf', gamma=g)
        
        labels = model.fit_predict(fpc_scores)
        
        # --- CÁLCULO DE MÉTRICAS ---
        results_temp = results_temp_base.copy()
        results_temp['label'] = labels
        
        real_outliers_info = results_temp[results_temp['label'] == -1]
        identified_unique = real_outliers_info.drop_duplicates(subset=['sub', 'repID'])
        
        matched_outliers = pd.merge(identified_unique, outliers_csv, on=['sub', 'repID'], how='inner')
        missing_outliers = pd.merge(outliers_csv, identified_unique, on=['sub', 'repID'], how='left', indicator=True)
        missing_outliers = missing_outliers[missing_outliers['_merge'] == 'left_only']
        
        TP = len(matched_outliers)
        FP = len(identified_unique) - TP
        FN = len(missing_outliers)
        
        if (TP + FP) > 0:
            precision = (TP / (TP + FP)) * 100
            recall = (TP / total_known_outliers) * 100
            f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            f2_score = (5 * precision * recall) / (4 * precision + recall) if (precision + recall) > 0 else 0
        else:
            precision, recall, f1_score, f2_score = 0.0, 0.0, 0.0, 0.0
            
        percentagem_lixo = (len(identified_unique) / total_trials) * 100
            
        results_log.append({
            'nu': nu,
            'gamma': g,
            'Recall': recall,
            'Precision': precision,
            'F1-Score': f1_score,
            'F2-Score': f2_score,
            'TP': TP,
            'FP': FP,
            'FN': FN,
            'Percentagem Lixo': percentagem_lixo
        })

# --- SELEÇÃO DO MELHOR MODELO OCSVM ---
results_df = pd.DataFrame(results_log)
limite_lixo = percentagem_conhecida + 5.0 
acceptable_models = results_df[results_df['Percentagem Lixo'] <= limite_lixo]

if acceptable_models.empty:
    acceptable_models = results_df

best_results = acceptable_models.sort_values(by=['Recall', 'F2-Score'], ascending=False)
best_params = best_results.iloc[0]

# CORREÇÃO: Os prints agora chamam as chaves certas do dicionário
print(f"\n🏆 VENCEDOR (One-Class SVM + fPCA):")
print(f"Nu: {best_params['nu']} | Gamma: {best_params['gamma']}")
print(f"F2-Score: {best_params['F2-Score']:.2f}% | Lixo: {best_params['Percentagem Lixo']:.2f}%")


--- Iniciando Grid Search Dinâmico (fPCA + One-Class SVM) ---

🏆 VENCEDOR (One-Class SVM + fPCA):
Nu: 0.119 | Gamma: scale
F2-Score: 51.14% | Lixo: 8.42%
